# 6 · Execution and impact

What happens between the decision and the fill.

A historical print tells you the price, never what your order would have
done to it. Here the same seed runs twice, once with your orders and once
without, so every fill is priced against the market where you never traded.

That counterfactual is what arrival price, VWAP and fitted impact models
approximate. Here it is measured directly.

In [1]:
import tradefloor as tf

universe = tf.Universe.random(20, seed=111)

execution = tf.tca.analyse(tf.baselines.Momentum(),
                           seed=7, universe=universe, days=3)

print("fills           :", len(execution.fills))
print("partial fills   :", len(execution.partial_fills()))
print("shortfall (bps) :", round(execution.shortfall_bps(), 3))

fills           : 137
partial fills   : 0
shortfall (bps) : 6.347


## The counterfactual run

`actual_final` is the closing price of each instrument. `baseline_final` is
the same market with the agent's orders removed: same seed, same draws,
everything else identical. Where they differ, that difference is the
agent's footprint.

In [2]:
tickers = [i.ticker for i in universe]
actual, baseline = execution.actual_final, execution.baseline_final

print(f"{'ticker':8s} {'with orders':>13s} {'without':>13s} {'difference':>12s}")
for t, a, b in zip(tickers, actual, baseline):
    if a != b:
        print(f"{t:8s} {a:13.4f} {b:13.4f} {a - b:12.4f}")

untouched = sum(1 for a, b in zip(actual, baseline) if a == b)
print(f"\n{untouched} of {len(tickers)} names were not moved at all.")

ticker     with orders       without   difference
AAA           199.7258      199.7259      -0.0000
AAB            11.8312       11.8311       0.0000
AAC            23.8218       23.8218       0.0000
AAD             6.5625        6.5626      -0.0001
AAE            18.5800       18.5800      -0.0000
AAF             4.3052        4.3052       0.0000
AAG           278.0100      278.0162      -0.0062
AAH             6.1355        6.1355       0.0000
AAI             7.5404        7.5404       0.0000
AAJ           130.6206      130.6206       0.0000
AAK             7.1511        7.1511      -0.0000
AAL            56.7251       56.7250       0.0001
AAM           103.8984      103.8987      -0.0004
AAN             4.0670        4.0669       0.0001
AAO            10.5746       10.5746      -0.0000
AAP           152.4230      152.4327      -0.0097
AAQ             9.6722        9.6722      -0.0000
AAR             4.7466        4.7454       0.0012
AAS             5.2203        5.2203       0.0000


## Which names moved

On pt-v20 all twenty closes differ, each by less than a cent. pt-v20 ends
the session with a closing cross at the model price instead of on the last
print, so a close is no longer rounded to the cent and a footprint too
small to move a print still shows. The agent traded all twenty names, and
the smallest footprint, on AAE, is under a millionth of a basis point. On pt-v19, which closed on a
print, 14 of the 20 closed on the same cent with and without the orders.

`moved()` reports every instrument whose final price differs between the two
runs, and it reports that difference in basis points, not in currency. The
column printed below as `price moved` is therefore a bps figure, so
it agrees with `impact_bps` to the rounding: both are the same measurement.
Bps is what lets a displacement on AAR, trading below 5, sit in the same
column as one on AAT, trading near 503. For the move in currency, read the
table above.

In [3]:
moved = execution.moved()
worst = sorted(moved.items(), key=lambda kv: -abs(kv[1]))[:6]

print(f"{'ticker':8s} {'price moved':>13s} {'impact (bps)':>14s}")
for ticker, delta in worst:
    print(f"{ticker:8s} {delta:13.4f} {execution.impact_bps(ticker):14.3f}")

ticker     price moved   impact (bps)
AAR             2.5427          2.543
AAP            -0.6360         -0.636
AAG            -0.2214         -0.221
AAN             0.2016          0.202
AAD            -0.1337         -0.134
AAM            -0.0353         -0.035


## Where the cost fell

`by_step` attributes the shortfall across decision points and `by_ticker`
across names. Positive is cost paid, and every step below is one. Until 0.8.5 some
steps here came out negative, fills better than the untraded market,
because the harness counted each order's flow on every tick of its step
and later trades sold back into that. An order's own impact is now smaller
than what the book charges to trade it, so there is nothing of it to sell
back into. Cost concentrated in a few steps says something about the
schedule.

In [4]:
steps = execution.by_step()
print("by step:")
for step, value in steps:
    bar = "#" * int(min(40, abs(value) / max(1, max(abs(v) for _, v in steps)) * 40))
    print(f"  step {step:3d} {value:12,.0f}  {bar}")

by step:
  step   6          537  ########################################
  step   7          526  #######################################
  step   8          348  #########################
  step   9          510  #####################################
  step  10          348  #########################
  step  11          373  ###########################
  step  12          218  ################
  step  13          172  ############
  step  14          254  ##################
  step  15          348  #########################
  step  16          289  #####################
  step  17           54  ####


In [5]:
by_ticker = execution.by_ticker()
top = sorted(by_ticker.items(), key=lambda kv: -abs(kv[1]))[:6]
print("largest contributions by name:")
for ticker, value in top:
    print(f"  {ticker:8s} {value:12,.0f}")

largest contributions by name:
  AAI               398
  AAS               385
  AAH               380
  AAP               354
  AAD               309
  AAN               254


## Partial fills

An order asking for more than the book holds at a price does not silently
receive it. Queue position and depth decide what you actually get.

In [6]:
if execution.partial_fills():
    print(f"{len(execution.partial_fills())} partial fills")
    for f in execution.partial_fills()[:5]:
        print("  ", f)
else:
    print("No partial fills at this size; the book absorbed every order.")
    print("Raising participation or size is how you find the edge of that;")
    print("the whole point is that the edge exists and is measurable.")

No partial fills at this size; the book absorbed every order.
Raising participation or size is how you find the edge of that;
the whole point is that the edge exists and is measurable.


## The same orders in a thinner book

The cell above says the edge exists and is measurable, and then does not
reach it: at this size the book absorbs everything. Raising participation
is one way to find the edge. Taking the depth away is the other, and it is
the one that matches what happens to you rather than what you chose.

A scenario is how you say that. `market.liquidity` scales the `avg_volume`
column the market maker quotes off, so every ladder level thins and the
same order walks further up the book. `macro.vix` widens the quote at the
same time, which is what a funding event does to both at once.

Nothing about the agent changes. Same seed, same market, same decisions --
only what it costs to act on them.


In [7]:
squeeze = (tf.Scenario(name="funding squeeze")
           .shock("market.liquidity", operation="multiply", value=0.35,
                  at=0, duration=3)
           .shock("macro.vix", operation="multiply", value=2.0,
                  at=0, duration=3))

print(f"{'book':10s} {'shortfall bps':>14s} {'fills':>7s} {'partials':>9s}")
for label, scenario in (("calm", None), ("squeeze", squeeze)):
    ex = tf.tca.analyse(tf.baselines.Momentum(), seed=7,
                        universe=universe, days=3, scenario=scenario)
    print(f"{label:10s} {ex.shortfall_bps():14.3f} {len(ex.fills):7d} "
          f"{len(ex.partial_fills()):9d}")


book        shortfall bps   fills  partials


calm                6.347     137         0


squeeze             9.283     144         0


The shortfall rises by nearly a half, from 6.35 to 9.28 bps: in the thin
book the same orders walk further for the same shares. None fills in part.
pt-v20 puts a latent book behind the maker's ladder (the `book_depth_*`
dials), so an order that outruns the ladder walks on at worse prices
instead of being cut off, and at this size the squeeze charges more rather
than refusing. On pt-v19, which has no depth past the ladder, the same run
went from 6.52 to 11.95 bps and four orders filled in part. That cost is
the whole reason this simulator prices execution rather than assuming
it. An evaluation that reads only the price series scores the two runs the
same, because the *prices* barely move -- the shock is in the book.

`tradefloor scenario list` names the scenarios that ship with the package;
`liquidity_crisis` is this shape on a longer horizon, with an assumed
credit response beside it. Load one with `tf.Scenario.load("...")`.

Its `transmission` block is worth reading before believing: those entries
are assumptions its author made, not effects this simulator derives.


## Comparing three algorithms

Same market, same seed, with the counterfactual computed for each.

In [8]:
candidates = {
    "momentum":       tf.baselines.Momentum(),
    "mean reversion": tf.baselines.MeanReversion(),
    "buy and hold":   tf.baselines.BuyAndHold(),
}

print(f"{'algorithm':16s} {'shortfall bps':>14s} {'fills':>7s} {'partials':>9s}")
for name, agent in candidates.items():
    ex = tf.tca.analyse(agent, seed=7, universe=universe, days=3)
    print(f"{name:16s} {ex.shortfall_bps():14.3f} {len(ex.fills):7d} "
          f"{len(ex.partial_fills()):9d}")

algorithm         shortfall bps   fills  partials


momentum                  6.347     137         0


mean reversion            6.619     138         0


buy and hold             10.571      20         0


## Provenance

An execution result carries the seed and model fingerprint, so a TCA number
can be cited.

In [9]:
print("seed             :", execution.seed)
print("model fingerprint:", execution.model_fingerprint)

seed             : 7
model fingerprint: pt-v20


## Caveats

**Volume changes were this notebook's structural caveat, and are not any
more.** A ceiling in the engine capped a name's volume response at a four
percent daily move (`volume_move_cap` 4.0 through `pt-v11`), so a violent day
traded like a quiet one. `pt-v12` raised the ceiling to twelve, and every
default since keeps it there, including `pt-v20`, the preset fingerprinted
above. On its record `volume_change_acf1` is in band at both horizons, and
the volume-change gap was retired from the realism envelope. `pt-v10` and
`pt-v11` are still selectable and still miss that row on their records, so a
study pinned to one of them keeps the old caveat. On the default,
schedule-shape conclusions no longer carry that warning, and depth, queue and
impact conclusions are sound as they were before. Notebook 04 prints the panel
this rests on. What the envelope still forbids lives in five other gaps
(horizon, decay-shape, scenario-magnitude, macro-range, roster-concentration),
none of them about the fill mechanics measured here.

**Single venue, no latency.** One book per name, orders arrive instantly,
and no strategic counterparties adapt to you.

Full documentation: <https://docs.tradefloor.dev>